# Lesson 17: Template Matching and Normalized Cross-Correlation

**Template matching** answers a simple question: where in a larger image does a small template image appear? The obvious approach &mdash; slide the template over the image and correlate at every position, like convolution in Lesson 9 &mdash; has a serious flaw. This lesson builds that up, exposes the flaw, and fixes it with **normalized cross-correlation (NCC)**.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## The naive approach: raw cross-correlation

At every position, compute the dot product between the template and the underlying image patch. A higher dot product should mean a better match &mdash; and if the template appears in the scene at the same brightness, this works fine.

In [ ]:
rng = np.random.default_rng(0)
template = rng.integers(0, 255, (15, 15)).astype(np.float64)

scene = rng.integers(50, 80, (60, 80)).astype(np.float64)
scene[20:35, 30:45] = template  # embed an exact copy

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(template, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Template')
axes[1].imshow(scene, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Scene (template embedded)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## The problem: brightness and contrast bias raw correlation

A dot product $\sum_{i,j} T(i,j)\,I(x+i, y+j)$ grows simply if the underlying patch is *brighter*, whether or not it actually resembles the template's pattern. We demonstrate: embed the exact template once (dim background) and a much *brighter* copy of the same template elsewhere. Raw correlation should be fooled into preferring the brighter copy.

In [ ]:
rng2 = np.random.default_rng(1)
bias_template = rng2.integers(80, 180, (20, 20)).astype(np.float32)

bias_scene = np.full((100, 120), 30, dtype=np.float32)
bias_scene[20:40, 20:40] = bias_template                                            # true match, dim scene
bias_scene[50:70, 70:90] = np.clip(bias_template.astype(np.float64) + 120, 0, 255)  # brightened copy elsewhere

raw_result = cv2.matchTemplate(bias_scene, bias_template, cv2.TM_CCORR)
raw_best = np.unravel_index(np.argmax(raw_result), raw_result.shape)

print(f'true match location:        (20, 20)')
print(f'raw cross-correlation picks: {raw_best}  <-- fooled by the brighter decoy')

plt.imshow(bias_scene, cmap='gray')
plt.title('True match (dim, top-left) vs. brighter decoy (bottom-right)')
plt.axis('off')
plt.show()

## The fix: normalized cross-correlation

NCC (in its zero-mean form, often called ZNCC) subtracts off each patch's own mean and divides by its own standard deviation before correlating &mdash; the same normalization trick used for a Pearson correlation coefficient in statistics:

$$\text{NCC}(x,y) = \frac{\sum_{i,j} \big(T(i,j)-\bar{T}\big)\big(I(x+i,y+j)-\bar{I}_{x,y}\big)}{\sqrt{\sum_{i,j}\big(T(i,j)-\bar{T}\big)^2} \; \sqrt{\sum_{i,j}\big(I(x+i,y+j)-\bar{I}_{x,y}\big)^2}}$$

where $\bar{T}$ is the template's mean and $\bar{I}_{x,y}$ is the mean of the image patch currently under the template. The result always lies in $[-1, 1]$: $1$ means a perfect match up to any brightness offset and any positive contrast scaling, regardless of the patch's absolute brightness.

In [ ]:
def normalized_cross_correlation(image, template):
    th, tw = template.shape
    h, w = image.shape
    t_zero = template - template.mean()
    t_norm = np.sqrt((t_zero**2).sum())

    result = np.zeros((h - th + 1, w - tw + 1))
    for y in range(result.shape[0]):
        for x in range(result.shape[1]):
            patch = image[y:y + th, x:x + tw]
            p_zero = patch - patch.mean()
            denom = np.sqrt((p_zero**2).sum()) * t_norm
            result[y, x] = (p_zero * t_zero).sum() / denom if denom > 1e-8 else 0
    return result

### Sanity check against OpenCV

`cv2.matchTemplate` with `TM_CCOEFF_NORMED` implements exactly this.

In [ ]:
mine = normalized_cross_correlation(scene, template)
reference = cv2.matchTemplate(scene.astype(np.float32), template.astype(np.float32), cv2.TM_CCOEFF_NORMED)

print(f'max abs difference: {np.abs(mine - reference).max():.2e}  (float32 vs float64 rounding only)')

best = np.unravel_index(np.argmax(mine), mine.shape)
print(f'best match at (row, col) = {best}, NCC score = {mine[best]:.4f}')

### NCC resists the brightness trap

Rerunning the decoy experiment with `TM_CCOEFF_NORMED` instead of raw `TM_CCORR`:

In [ ]:
ncc_result = cv2.matchTemplate(bias_scene, bias_template, cv2.TM_CCOEFF_NORMED)
ncc_best = np.unravel_index(np.argmax(ncc_result), ncc_result.shape)

print(f'true match location:  (20, 20)')
print(f'NCC picks:            {ncc_best}')
print(f'NCC score at true match: {ncc_result[20, 20]:.4f}')
print(f'NCC score at decoy:      {ncc_result[50, 70]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(raw_result, cmap='viridis')
axes[0].set_title('Raw correlation map\n(peaks at bright decoy)')
axes[1].imshow(ncc_result, cmap='viridis')
axes[1].set_title('NCC map\n(peaks at true match)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Both locations contain the *same underlying pattern*, just at different brightness &mdash; and NCC correctly reports comparably high scores at both (since it's contrast/brightness-invariant), while still correctly ranking the true, less-altered match highest and, unlike raw correlation, never getting fooled into thinking brightness alone is a stronger cue than pattern similarity.

## Localizing matches with `cv2.minMaxLoc`

For real use, `cv2.minMaxLoc` finds the best score and its location directly (avoiding a manual `argmax` + `unravel_index`), and the match's bounding box follows immediately from the template's size.

In [ ]:
scene_color = cv2.cvtColor(scene.astype(np.uint8), cv2.COLOR_GRAY2BGR)
result = cv2.matchTemplate(scene.astype(np.float32), template.astype(np.float32), cv2.TM_CCOEFF_NORMED)
_, max_val, _, max_loc = cv2.minMaxLoc(result)

th, tw = template.shape
top_left = max_loc
bottom_right = (top_left[0] + tw, top_left[1] + th)
cv2.rectangle(scene_color, top_left, bottom_right, (0, 0, 255), 1)

print(f'best match score: {max_val:.4f} at top-left corner {top_left}')
plt.imshow(cv2.cvtColor(scene_color, cv2.COLOR_BGR2RGB))
plt.title('Located match')
plt.axis('off')
plt.show()

## What NCC does *not* fix

NCC is invariant to a linear (affine) change in intensity &mdash; brightness offset and positive contrast scaling. It is **not** invariant to rotation or scale: a template searched for at the wrong size or orientation will simply not match well anywhere, because template matching only ever slides the template around, it never rotates or resizes it. Handling that requires either searching over many rotated/scaled copies of the template (expensive), or switching to the kind of scale- and rotation-invariant local features covered in a later lesson.

### Exercise

1. Rotate `template` by 30 degrees (Lesson 7's `cv2.warpAffine`) before embedding it in a copy of `scene`, then run `TM_CCOEFF_NORMED` against the *unrotated* template. How much does the peak NCC score drop compared to the unrotated case?
2. Try `cv2.TM_SQDIFF_NORMED` (normalized sum of squared differences) instead of `TM_CCOEFF_NORMED` on the brightness-decoy scene. Note that for this method, the *best* match is the *minimum* value, not the maximum &mdash; find it correctly and check whether it also resists the brightness trap.
3. Add heavy Gaussian noise to `scene` (Lesson 13's noise model) and re-run the search. At what noise level does the NCC peak stop reliably landing on the true embedded location?